# Homework-3 

## Use the Titanic Dataset to Practice Data Wrangling Tasks
### Part 1: Import libaries , load the data, inspect the data, then get the name of the variables and create a small markdown table  with variable names and types (e.g numerical or categorical ordinal, categorical nominal etc.)

### Part 2: 
    - Impute missing values for "Embarked" with the mode (most frequent value)
    - Impute missing value for Age using two following strategies
        - A) Simple mean imputation (fill with the overal mean age)
        - B) Group-wise median imputation: for each combination of Sex and Pclass compute the median age. Impute missing age using the median age of passenger's sex and Pclass group

### Part 3: Create a FamilySize variable (FamilySize = SibSp + Parch +1) and compute:
    - Minimum, Maximum, and mean FamilySize
    - How many passengers were traveling alone
    
### Part 4:  Create a filtered dataframe of all passengers:
    - Who are female
    - Who are male 
    - Who are under 16 years old   
    - Compute the survival rate for the above filtered data
    - Compute survival rate by Sex and Pclass (use groupby and mean) and present the result in a markedown table (rows: sex, columns: Pclass) or using print formating.
    - Interpret your result in 2-3 sentences
     
    
### Part 5: Compute the average fare for each Pclass (Fare values are per tickets not per person (there coud be more than 1 person on a ticket) so make sure to account for that. 

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df = pd.read_csv('titanic.csv')

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### Titanic Dataset Variables
| Variable Name | Data Type | Description |
| :--- | :--- | :--- |
| **PassengerId** | Numerical (Discrete) | Unique ID for each passenger |
| **Survived** | Categorical (Nominal) | 0 = No, 1 = Yes |
| **Pclass** | Categorical (Ordinal) | Ticket class (1, 2, 3) |
| **Name** | Categorical (Nominal) | Passenger name |
| **Sex** | Categorical (Nominal) | Male or Female |
| **Age** | Numerical (Continuous) | Age in years |
| **SibSp** | Numerical (Discrete) | # of siblings/spouses aboard |
| **Parch** | Numerical (Discrete) | # of parents/children aboard |
| **Ticket** | Categorical (Nominal) | Ticket number |
| **Fare** | Numerical (Continuous) | Passenger fare |
| **Cabin** | Categorical (Nominal) | Cabin number |
| **Embarked** | Categorical (Nominal) | Port of Embarkation (C, Q, S) |

In [18]:
mode_embarked = df['Embarked'].mode()[0]

df['Embarked'] = df['Embarked'].fillna(mode_embarked)



In [19]:
df['Age'] = df.groupby(['Sex', 'Pclass'])['Age'].transform(lambda x: x.fillna(x.median()))

df.isnull().sum()

PassengerId          0
Survived             0
Pclass               0
Name                 0
Sex                  0
Age                  0
SibSp                0
Parch                0
Ticket               0
Fare                 0
Cabin              687
Embarked             0
FamilySize           0
TicketGroupSize      0
FarePerPerson        0
dtype: int64

In [20]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df[['SibSp', 'Parch', 'FamilySize']].head()

,SibSp,Parch,FamilySize
0,1,0,2
1,1,0,2
2,0,0,1
3,1,0,2
4,0,0,1


In [21]:
fam_min = df['FamilySize'].min()
fam_max = df['FamilySize'].max()
fam_mean = df['FamilySize'].mean()

print(f"Minimum Family Size: {fam_min}")
print(f"Maximum Family Size: {fam_max}")
print(f"Average Family Size: {fam_mean:.2f}")

Minimum Family Size: 1
Maximum Family Size: 11
Average Family Size: 1.90


In [22]:
alone_count = (df['FamilySize'] == 1).sum()

print(f"Number of passengers traveling alone: {alone_count}")

Number of passengers traveling alone: 537


In [23]:
females = df[df['Sex'] == 'female']
males = df[df['Sex'] == 'male']
under_16 = df[df['Age'] < 16]

print(f"Number of females: {len(females)}")
print(f"Number of males: {len(males)}")
print(f"Number of children (under 16): {len(under_16)}")

Number of females: 314
Number of males: 577
Number of children (under 16): 83


In [24]:

female_survival = females['Survived'].mean()
male_survival = males['Survived'].mean()
child_survival = under_16['Survived'].mean()

print(f"Female Survival Rate: {female_survival:.2%}")
print(f"Male Survival Rate: {male_survival:.2%}")
print(f"Under 16 Survival Rate: {child_survival:.2%}")

Female Survival Rate: 74.20%
Male Survival Rate: 18.89%
Under 16 Survival Rate: 59.04%


In [25]:

survival_table = df.groupby(['Sex', 'Pclass'])['Survived'].mean().unstack()

print("Survival Rates by Sex and Class:")
print(survival_table)

Survival Rates by Sex and Class:
Pclass         1         2         3
Sex                                 
female  0.968085  0.921053  0.500000
male    0.368852  0.157407  0.135447


### the data over here shows that the women and  children were the real trend and it would look like 74% females and kids 59% for kids under 16 had much more of a higher survival rate then male. it would seem that social class also played some sort role with the passangers having a higher chance at suriving compared to the other class.
 


In [26]:
ticket_counts = df['Ticket'].value_counts()

df['TicketGroupSize'] = df['Ticket'].map(ticket_counts)

df['FarePerPerson'] = df['Fare'] / df['TicketGroupSize']



In [27]:
avg_fare_per_class = df.groupby('Pclass')['FarePerPerson'].mean()

print("Average Fare per Person by Class:")
print(avg_fare_per_class)

Average Fare per Person by Class:
Pclass
1    43.650347
2    13.322599
3     8.085857
Name: FarePerPerson, dtype: float64


In [28]:
comparison = df.groupby('Pclass')[['Fare', 'FarePerPerson']].mean()
print(comparison)

             Fare  FarePerPerson
Pclass                          
1       84.154687      43.650347
2       20.662183      13.322599
3       13.675550       8.085857
